**🔗 Live Deployment Link:** https://YOUR-APP-NAME.streamlit.app/  *(replace with your real link)*

# Heart Disease Prediction System
**Course:** AIC354 Machine Learning Fundamentals | **Assignment 2, Task 2**
**Name / Reg. No.:** _your name and registration number_

**Problem:** Binary classification, predict whether a patient has heart disease (1) or not (0).
**Dataset:** UCI Heart Disease (Cleveland), https://archive.ics.uci.edu/dataset/45/heart+disease
**Selected 4 input attributes:** `age`, `sex`, `cp` (chest pain type), `thalach` (max heart rate achieved)
**Model:** Support Vector Classifier (same approach as the Titanic system)

Steps: Import Libraries → Load Data → Understand Data → Preprocess → Encode → Train → Test → Save/Deploy → Feedback

## Step 1: Import Libraries

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## Step 2: Data Collection (Load Data)
The UCI file `processed.cleveland.data` has no header row, so we supply the column names ourselves.
Missing values are written as `?`, so we tell pandas to treat `?` as NaN.
(Place the file next to this notebook, renamed to `processed_cleveland.csv`.)

In [27]:
columns = ["age","sex","cp","trestbps","chol","fbs","restecg",
           "thalach","exang","oldpeak","slope","ca","thal","num"]

df = pd.read_csv("processed_cleveland.csv", names=columns, na_values="?")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## Step 3: Data Understanding

In [28]:
print("Shape:", df.shape)
df.info()

Shape: (303, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
df.describe()

In [ ]:
# 'num' is 0 (no disease) or 1-4 (disease present severity). We simplify to binary.
print(df["num"].value_counts().sort_index())

## Step 4: Data Preprocessing
1. Create the binary target `target` (0 = no disease, 1 = disease).
2. Keep only the 4 chosen input attributes (assignment constraint).
3. Handle missing values and duplicates.
4. Convert coded categories to readable text so that label encoding is meaningful.

In [ ]:
df["target"] = (df["num"] > 0).astype(int)

data = df[["age", "sex", "cp", "thalach", "target"]].copy()

print("Missing before:", data.isnull().sum().sum())
data = data.dropna()
print("Missing after :", data.isnull().sum().sum())
print("Duplicate rows:", data.duplicated().sum(), "(kept: different patients can share values)")

# readable categories
data["sex"] = data["sex"].map({0: "Female", 1: "Male"})
data["cp"] = data["cp"].map({1: "Typical Angina", 2: "Atypical Angina",
                             3: "Non-anginal Pain", 4: "Asymptomatic"})
data.head()

### Quick visualization (EDA)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
data.groupby("sex")["target"].mean().plot(kind="bar", ax=ax[0], title="Disease rate by Sex", rot=0)
data.groupby("cp")["target"].mean().plot(kind="bar", ax=ax[1], title="Disease rate by Chest Pain", rot=20)
data.boxplot(column="thalach", by="target", ax=ax[2]); ax[2].set_title("Max Heart Rate by Target")
plt.suptitle(""); plt.tight_layout(); plt.show()

## Step 5: Label Encoding
Models need numbers, not text. `LabelEncoder` turns each category into an integer.
We keep the encoders so the deployed app can encode new input in exactly the same way.

In [ ]:
le_sex = LabelEncoder()
le_cp = LabelEncoder()

data["sex"] = le_sex.fit_transform(data["sex"])
data["cp"] = le_cp.fit_transform(data["cp"])

print("sex mapping:", dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_))))
print("cp mapping :", dict(zip(le_cp.classes_, le_cp.transform(le_cp.classes_))))
data.head()

## Step 6: Train the Model
Split 80% train / 20% test, then train an SVC (with scaling, since `age` and `thalach` are on a different scale from the encoded columns).

In [ ]:
X = data[["age", "sex", "cp", "thalach"]]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

model = Pipeline([("scaler", StandardScaler()),
                  ("svc", SVC(kernel="rbf", C=1.0, random_state=42))])
model.fit(X_train, y_train)

## Step 7: Test the Model

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["No Disease", "Disease"]).plot()
plt.show()

### Predict for a new patient

In [ ]:
def predict_patient(age, sex, cp, thalach):
    row = pd.DataFrame([[age, le_sex.transform([sex])[0], le_cp.transform([cp])[0], thalach]],
                       columns=["age", "sex", "cp", "thalach"])
    p = model.predict(row)[0]
    return "Heart disease likely" if p == 1 else "Heart disease not likely"

print(predict_patient(58, "Male", "Asymptomatic", 120))
print(predict_patient(35, "Female", "Non-anginal Pain", 175))

## Step 8: Save the Model (for Deployment)

In [ ]:
joblib.dump({"model": model, "le_sex": le_sex, "le_cp": le_cp}, "heart_disease_model.pkl")
print("Saved heart_disease_model.pkl")

# check it reloads and predicts without retraining
saved = joblib.load("heart_disease_model.pkl")
print("Reload test accuracy:", round(accuracy_score(y_test, saved["model"].predict(X_test)), 3))

## Step 9: Cloud Deployment
The model is deployed with **Streamlit Community Cloud** (same as the Titanic demo) using `app.py`, `requirements.txt` and `heart_disease_model.pkl` from the GitHub repository.

Live link: https://YOUR-APP-NAME.streamlit.app/

## Step 10: Feedback / Conclusion
- The model uses only 4 features, so accuracy is modest. Features such as `thal`, `ca` and `oldpeak` are strong predictors and would likely improve results.
- Because this is a medical-style problem, **recall** for the "Disease" class matters most: missing a sick patient is worse than a false alarm.
- Future work: try more features, compare Logistic Regression / Decision Tree, and use cross-validation.
- This system is for learning only and is not a medical diagnosis tool.